# 13 — Industry Domain Agents (real model decisions, real eval)

Three of this repo's own case-study domains (BFSI fraud triage, Legal contract review, Retail
order-issue triage), each rebuilt so the real model — not a scripted policy — makes the actual
decision, then real-vs-expected accuracy is reported.

**Honest scope**: like the repo's own GPU-free case studies, the tool outputs (transaction
velocity, clause terms, order status) are deterministic stand-ins for real risk/CLM/OMS
services — there is no small open real dataset for "flagged card transactions" or "contract
clauses" to plug in here. What is real and NOT scripted: the model reads those tool outputs and
makes the triage/review/routing decision itself, scored against expected labels — unlike the
original case studies, where the decision itself was a canned `if/else`.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available()
print(f"[gpu] {torch.cuda.get_device_name(0)} | torch {torch.__version__}")

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")

def decide(system, user, max_new_tokens=8):
    enc = tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                  add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()

def run_domain(name, system, cases):
    correct = 0
    for user, expected in cases:
        raw = decide(system, user)
        verdict = next((v for v in expected.split("|") if v in raw), raw.split()[0] if raw else "")
        ok = verdict in expected.split("|")
        correct += int(ok)
        print(f"[{name}] expected={expected:<20} decided={raw[:20]!r:<22} {'OK' if ok else 'MISS'}")
    print(f"[{name}] real accuracy: {correct}/{len(cases)}\n")
    return correct, len(cases)

[gpu] NVIDIA GeForce RTX 4090 | torch 2.11.0+cu130


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 1) BFSI — fraud triage (APPROVE / REVIEW / BLOCK)

Same real deterministic tool stand-ins as this repo's own `fraud_triage_end_to_end.py` example
(velocity + geo checks) — but here the real model reads their output and decides, instead of a
canned `if/else`.

In [2]:
def check_velocity(txn_per_hour, threshold=5):
    return {"txn_per_hour": txn_per_hour, "threshold": threshold, "over": txn_per_hour > threshold}

def check_geo(home, txn_geo):
    return {"home": home, "txn_geo": txn_geo, "mismatch": home != txn_geo}

FRAUD_SYS = ("You are a fraud analyst. Given velocity and geo check results, decide exactly one "
            "of: APPROVE, REVIEW, BLOCK. Reply with ONLY that one word.")

fraud_cases = []
for velocity, geo, expected in [
    ({"txn_per_hour": 2, "threshold": 5}, {"home": "US-TX", "txn_geo": "US-TX"}, "APPROVE"),
    ({"txn_per_hour": 9, "threshold": 5}, {"home": "US-TX", "txn_geo": "RO-B"}, "BLOCK|REVIEW"),
    ({"txn_per_hour": 6, "threshold": 5}, {"home": "US-TX", "txn_geo": "US-TX"}, "REVIEW|APPROVE"),
]:
    v = check_velocity(**velocity)
    g = check_geo(**geo)
    fraud_cases.append((f"velocity={v}, geo={g}", expected))

run_domain("fraud", FRAUD_SYS, fraud_cases)

[fraud] expected=APPROVE              decided='BLOCK'                MISS
[fraud] expected=BLOCK|REVIEW         decided='BLOCK'                OK
[fraud] expected=REVIEW|APPROVE       decided='BLOCK'                MISS
[fraud] real accuracy: 1/3



(1, 3)

## 2) Legal — contract clause review (ACCEPT / NEGOTIATE / REJECT)

Same real deterministic tool stand-ins as this repo's own `contract_review_end_to_end.py` example.

In [3]:
LEGAL_SYS = ("You are a contract reviewer. Given a liability clause and obligation check, "
            "decide exactly one of: ACCEPT, NEGOTIATE, REJECT. Reply with ONLY that one word.")

legal_cases = [
    ("clause: liability cap = 12 months fees, mutual. obligations: indemnifies=True, no uncapped carveouts.",
     "ACCEPT"),
    ("clause: liability cap = 3 months fees, one-sided. obligations: indemnifies=False, "
     "uncapped carveouts=[IP, confidentiality].", "NEGOTIATE"),
    ("clause: liability cap = NONE (unlimited liability for us only). obligations: indemnifies=False.",
     "REJECT|NEGOTIATE"),
]
run_domain("legal", LEGAL_SYS, legal_cases)

[legal] expected=ACCEPT               decided='REJECT'               MISS


[legal] expected=NEGOTIATE            decided='NEGOTIATE'            OK
[legal] expected=REJECT|NEGOTIATE     decided='REJECT'               OK
[legal] real accuracy: 2/3



(2, 3)

## 3) Retail — order-issue triage (WISMO / REFUND / REPLACEMENT)

Same labeled set as this repo's own `order_issue_triage.py` example — but the real model
reads the shopper's message and decides, instead of the original's keyword-matching
`route_for()` (which was the deterministic ground-truth generator, not a fair model decision).

In [4]:
RETAIL_SYS = ("You are a retail support triage agent. Given the shopper's message, decide "
             "exactly one of: WISMO, REFUND, REPLACEMENT. Reply with ONLY that one word.")

retail_cases = [
    ("where is my order, it hasn't arrived yet", "WISMO"),
    ("the mug arrived damaged, it's broken", "REPLACEMENT"),
    ("please refund me, I was charged twice", "REFUND"),
]
run_domain("retail", RETAIL_SYS, retail_cases)

[retail] expected=WISMO                decided='REFUND'               MISS


[retail] expected=REPLACEMENT          decided='REFUND'               MISS
[retail] expected=REFUND               decided='REFUND'               OK
[retail] real accuracy: 1/3



(1, 3)

## Summary

| Domain | Real evidence |
|---|---|
| BFSI fraud triage | real per-case decision + accuracy printed above |
| Legal contract review | real per-case decision + accuracy printed above |
| Retail order triage | real per-case decision + accuracy printed above |

In every domain the real `Qwen2.5-0.5B-Instruct` reads the (deterministic-stand-in) tool
outputs and makes the actual triage decision — the decision itself is never scripted, unlike
the GPU-free case studies this notebook is based on.